# Illustrated Losses And Metrics

A loss is optimized during training. A metric is reported to describe quality. Losses return tensors and can participate in gradients; metrics return JavaScript numbers.

Use the learning loop here too: predict what a perfect prediction would score, run the example, then interpret whether bigger or smaller is better.


In [ ]:
import { tensor, accuracy, precision, recall, f1, mse, mae, r2 } from 'affon:compute'
import nn from 'affon:nn'

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}


## Regression Losses And Metrics

$$
\mathrm{MSE}=\frac{1}{N}\sum_i(\hat{y}_i-y_i)^2
$$

$$
\mathrm{MAE}=\frac{1}{N}\sum_i|\hat{y}_i-y_i|
$$

`MSELoss` is the training loss version. `mse` and `mae` are reporting helpers.


### Before You Run

For predictions `[2, 4]` and targets `[1, 5]`, each error has magnitude `1`. Predict MSE and MAE before running.

### Read The Output

MSE and MAE are both `1` here because both mistakes have absolute error `1`; squaring does not change them.


In [ ]:
const pred = tensor([2, 4])
const target = tensor([1, 5])
const mseLoss = nn.MSELoss()
show('MSELoss(pred, target)', mseLoss(pred, target))
console.log('mse:', mse(pred, target))
console.log('mae:', mae(pred, target))
console.log('r2:', r2(pred, target))


## Binary Losses

`BCELoss` expects probability-like predictions. `BCEWithLogitsLoss` expects raw logits and applies the sigmoid-style conversion internally.

```txt
probability -> BCELoss
logit       -> BCEWithLogitsLoss
```


### Before You Run

Ask which input is already probability-like and which input is a raw logit. That decides whether to use `BCELoss` or `BCEWithLogitsLoss`.


In [ ]:
const binaryTarget = tensor([1, 0, 1])
show('BCELoss probabilities', nn.BCELoss()(tensor([0.9, 0.2, 0.7]), binaryTarget))
show('BCEWithLogitsLoss logits', nn.BCEWithLogitsLoss()(tensor([2, -1, 1]), binaryTarget))


## Cross Entropy

Cross entropy compares class logits against target class labels.

```txt
logits: [batch, classes]
target: [batch]
loss:   [1]
```

It rewards high score on the correct class and penalizes high scores on wrong classes.


### Before You Run

For each row of logits, identify the largest class score. If it matches the target class, the loss should be relatively small.


In [ ]:
const logits = tensor([[2, 1, 0], [0, 3, 1]])
const classes = tensor([0, 1], { dtype: 'i64' })
show('CrossEntropyLoss(logits, classes)', nn.CrossEntropyLoss()(logits, classes))


## Classification Metrics

Classification metrics are easiest to read through a confusion matrix.

<svg viewBox="0 0 560 250" width="560" height="250" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:16px monospace}.h{fill:#dbeafe;stroke:#60a5fa}.good{fill:#dcfce7;stroke:#4ade80}.bad{fill:#fee2e2;stroke:#f87171}</style>
  <text class="txt" x="210" y="30">actual</text><text class="txt" x="25" y="125">predicted</text>
  <rect class="h" x="170" y="55" width="130" height="40"/><text class="mono" x="220" y="80">yes</text>
  <rect class="h" x="300" y="55" width="130" height="40"/><text class="mono" x="350" y="80">no</text>
  <rect class="h" x="80" y="95" width="90" height="55"/><text class="mono" x="108" y="128">yes</text>
  <rect class="h" x="80" y="150" width="90" height="55"/><text class="mono" x="112" y="183">no</text>
  <rect class="good" x="170" y="95" width="130" height="55"/><text class="mono" x="222" y="128">TP</text>
  <rect class="bad" x="300" y="95" width="130" height="55"/><text class="mono" x="352" y="128">FP</text>
  <rect class="bad" x="170" y="150" width="130" height="55"/><text class="mono" x="222" y="183">FN</text>
  <rect class="good" x="300" y="150" width="130" height="55"/><text class="mono" x="352" y="183">TN</text>
</svg>

$$
precision = \frac{TP}{TP+FP},\quad recall = \frac{TP}{TP+FN}
$$


### Before You Run

Threshold the binary predictions at `0.5`. Count TP, FP, FN, and TN by hand, then compare with precision, recall, and F1.

### Debugging Cue

If accuracy looks good but precision or recall looks bad, inspect the confusion matrix. Accuracy can hide class imbalance.


In [ ]:
const binPred = tensor([0.9, 0.2, 0.8, 0.4])
const binTarget = tensor([1, 0, 0, 1])
console.log('precision:', precision(binPred, binTarget))
console.log('recall:', recall(binPred, binTarget))
console.log('f1:', f1(binPred, binTarget))
console.log('multiclass accuracy:', accuracy(logits, classes))
